In [34]:
from pyspark import SparkConf, SparkContext
import itertools
from math import sin, cos, radians, sqrt, asin

conf = SparkConf().setAppName('Lab1').setMaster('local')
sc = SparkContext.getOrCreate(conf=conf)

tripsFile = sc.textFile('trip.csv')
tripsIdx = tripsFile.first().split(',')
tripsData = tripsFile.mapPartitionsWithIndex(lambda idx, it: itertools.islice(it,1,None) if (idx==0) else it).map(lambda x: x.split(','))

stationsFile = sc.textFile('station.csv')
stationsIdx = stationsFile.first().split(',')
stationsData = stationsFile.mapPartitionsWithIndex(lambda idx, it: itertools.islice(it,1,None) if (idx==0) else it).map(lambda x: x.split(','))

1. Найти велосипед с максимальным временем пробега.

In [35]:
bike_id = 8
print(tripsIdx[bike_id])
duration_id = 1
print(tripsIdx[duration_id])

result = tripsData.map(lambda x: (str(x[bike_id]), int(x[duration_id]))).reduceByKey(lambda a, b: a + b).sortBy(lambda x: x[1], ascending=False).first()
print('Result: ', result)
selected_id = result[0]

bike_id
duration
Result:  ('535', 18611693)


2. Найти наибольшее геодезическое расстояние между станциями.

In [36]:
def Haversine(lat1, lon1, lat2, lon2):
  radius = 6371
  dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
  a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
  return 2 * radius * asin(sqrt(a))

id_id = 0
print(stationsIdx[id_id])
lat_id = 2
print(stationsIdx[lat_id])
lon_id = 3
print(stationsIdx[lon_id])

raw_data = stationsData.map(lambda x: (int(x[id_id]), (float(x[lat_id]), float(x[lon_id]))))
station_pairs = raw_data.cartesian(raw_data).filter(lambda x: x[0][0] > x[1][0] )
result = station_pairs.map(lambda x: ((x[0][0], x[1][0]), Haversine(x[0][1][0], x[0][1][1], x[1][1][0], x[1][1][1]))).sortBy(lambda x: x[1], ascending=False).first()
print('Result: ', result)

id
lat
long
Result:  ((60, 16), 69.9208759542826)


3. Найти путь велосипеда с максимальным временем пробега через станции.

In [37]:
bike_id = 8
print(tripsIdx[bike_id])
from_id = 3
print(tripsIdx[from_id])
to_id = 6
print(tripsIdx[to_id])
time_id = 2
print(tripsIdx[time_id])

result = tripsData.filter(lambda x: str(x[bike_id]) == selected_id).sortBy(lambda x: x[time_id], ascending=True).map(lambda x: (x[from_id], x[to_id])).collect()
print('Result: ', result)

bike_id
start_station_name
end_station_name
start_date
Result:  [('Mechanics Plaza (Market at Battery)', 'Embarcadero at Sansome'), ('Embarcadero at Sansome', 'Market at 4th'), ('Market at 4th', 'South Van Ness at Market'), ('Market at 10th', 'Powell Street BART'), ('Embarcadero at Folsom', 'San Francisco Caltrain (Townsend at 4th)'), ('San Francisco Caltrain (Townsend at 4th)', 'Temporary Transbay Terminal (Howard at Beale)'), ('Temporary Transbay Terminal (Howard at Beale)', 'Market at 10th'), ('Powell Street BART', 'Market at 10th'), ('Market at 10th', 'Market at 4th'), ('Market at 4th', 'San Francisco Caltrain (Townsend at 4th)'), ('Davis at Jackson', 'Beale at Market'), ('Beale at Market', 'Davis at Jackson'), ('San Francisco Caltrain (Townsend at 4th)', 'Embarcadero at Vallejo'), ('San Francisco Caltrain (Townsend at 4th)', 'Market at Sansome'), ('Market at Sansome', 'Davis at Jackson'), ('Howard at 2nd', '2nd at South Park'), ('2nd at South Park', '2nd at Folsom'), ('2nd at Fols

4. Найти количество велосипедов в системе.

In [38]:
bike_id = 8
print(tripsIdx[bike_id])

result = tripsData.map(lambda x: x[bike_id]).distinct().count()
print('Result: ', result)

bike_id
Result:  700


5. Найти пользователей потративших на поездки более 3 часов.

In [39]:
zip_id = 10
print(tripsIdx[zip_id])
duration_id = 1
print(tripsIdx[duration_id])

result = tripsData.map(lambda x: (str(x[zip_id]), int(x[duration_id]))).reduceByKey(lambda a, b: a + b).filter(lambda x: x[1] > 3600 * 3).collect()
print('Result: ', result)

zip_code
duration
Result:  [('95060', 758576), ('95112', 12742370), ('94041', 6276284), ('94117', 6901313), ('94402', 3303647), ('94102', 19128021), ('94612', 1860796), ('94609', 2503133), ('94158', 6248167), ('94133', 21637675), ('94597', 680583), ('', 27723273), ('94121', 2363342), ('95118', 1401452), ('94610', 3630628), ('95136', 1114542), ('2142', 15710), ('94703', 1704079), ('95070', 717380), ('94404', 3589350), ('94518', 481470), ('94549', 1929161), ('94556', 441764), ('94805', 64177), ('95014', 2366901), ('97330', 44364), ('94005', 258055), ('92178', 14156), ('85008', 21626), ('94606', 1066328), ('94941', 1837244), ('94901', 1528653), ('94577', 805757), ('94523', 451757), ('92111', 46914), ('95618', 79042), ('89052', 118993), ('94014', 1762637), ('10025', 217811), ('78230', 89117), ('10022', 439169), ('95111', 1340498), ('75201', 44995), ('94141', 15476), ('90046', 266665), ('34110', 39106), ('1945', 68179), ('75225', 40261), ('90032', 82923), ('4517', 46394), ('94080', 1002049)